# LeetCode 3995 — Minimum Cost to Convert String III bằng Python

- Link: https://leetcode.com/problems/minimum-cost-to-convert-string-iii/
- Độ khó: **Hard**
- Chủ đề: Quy hoạch động (DP) trên chuỗi, Khớp mẫu có ký tự đại diện (wildcard), Đường đi ngắn nhất trên DAG, Bitset

Skeleton trên LeetCode (Python3):

```python
class Solution:
    def minimumCost(self, source: str, target: str, rules: List[List[str]], costs: List[int]) -> int:
```

## 1. Giới thiệu & mục tiêu học

Cho chuỗi `source` và `target` cùng độ dài, cùng một danh sách **luật** `[mẫu, thay_thế]` có chi phí cơ sở. Một luật có thể áp dụng vào một đoạn của chuỗi hiện tại nếu mẫu **khớp** đoạn đó (`'*'` khớp ký tự bất kỳ) và **không ký tự nào của đoạn đã bị dùng ở lần áp dụng trước**; khi đó đoạn bị thay bằng chuỗi `thay_thế` (cùng độ dài). Chi phí = chi phí cơ sở + số dấu `'*'` của mẫu. Tìm tổng chi phí nhỏ nhất để biến `source` thành `target`, hoặc `-1`.

Quy tắc "mỗi vị trí chỉ dùng **một lần**" là chìa khoá: mỗi luật chỉ ghi lên các ô **chưa đụng tới**, nên khi nó khớp thì các ô đó vẫn còn nguyên là `source`. Vì vậy các lần áp dụng không chồng lên nhau và không phụ thuộc thứ tự → bài toán thành **chia `source` thành các đoạn rời nhau**, mỗi đoạn hoặc giữ nguyên (nếu đã bằng `target`) hoặc được một luật viết lại đúng thành đoạn `target` tương ứng. Đó là một DP một chiều trên chỉ số.

Sau notebook này bạn sẽ:

1. Đọc kỹ luật đề và mô phỏng trực tiếp (Dijkstra trên trạng thái chuỗi + mặt nạ ô đã dùng) làm trọng tài.
2. Rút gọn thành DP trên chỉ số `dp[i]` = chi phí nhỏ nhất để chuẩn bị xong `i` ký tự đầu.
3. Viết khớp mẫu có `'*'` bằng vòng lặp, bằng `re`, và bằng **bitset số nguyên** (rất nhanh).
4. Tách phần dựng cạnh (luật nào dùng được tại vị trí nào) khỏi phần DP.
5. Đối chiếu các cách bằng test ngẫu nhiên có seed và đo thời gian ở cỡ `n = 5000`, `200` luật.

## 2. Đề bài & ví dụ

**Phát biểu lại.**

- Cho `source`, `target` (cùng độ dài `n`), `rules[i] = [pattern_i, replacement_i]` (cùng độ dài), `costs[i]`.
- Mỗi lần áp dụng một luật: chọn chỉ số bắt đầu `l` sao cho đoạn `[l, l + len - 1]` nằm trong chuỗi và **chưa ô nào bị dùng ở lần trước**; ở mỗi vị trí `j`, `pattern[j]` bằng ký tự hiện tại hoặc là `'*'`; thay đoạn bằng `replacement` (không chứa `'*'`).
- Chi phí lần áp dụng: `costs[i] + (số '*' trong pattern_i)`.
- Ô đã dùng thì không dùng lại. Ô **không dùng** thì giữ nguyên.
- Trả về chi phí tổng nhỏ nhất để chuỗi cuối **bằng `target`**, hoặc `-1`.

**Ví dụ 1:** `source="hello"`, `target="world"`, `rules=[["he","wo"],["llo","rld"]]`, `costs=[3,4]` → `7`.

**Ví dụ 2:** `source="cat"`, `target="dog"`, `rules=[["c*t","dog"]]`, `costs=[2]` → `3` (`'*'` khớp `'a'`, +1).

**Ví dụ 3:** `source="test"`, `target="next"`, `rules=[["*e*t","next"]]`, `costs=[4]` → `6` (hai dấu `'*'`).

**Ví dụ 4:** `source="ab"`, `target="bc"`, `rules=[["a*","bd"]]`, `costs=[9]` → `-1` (luật khớp `"ab"` nhưng viết thành `"bd"`, không phải `"bc"`).

**Ràng buộc**

- `1 <= source.length == target.length <= 5000`
- `1 <= rules.length == costs.length <= 200`
- `1 <= pattern_i.length == replacement_i.length <= 20`; `pattern_i` có ít nhất 1 chữ cái, tối đa 5 dấu `'*'`; `replacement_i` chỉ chữ thường
- `1 <= costs[i] <= 1000`

In [1]:
from typing import List

EXAMPLES = [
    ("hello", "world", [["he", "wo"], ["llo", "rld"]], [3, 4], 7),
    ("cat", "dog", [["c*t", "dog"]], [2], 3),
    ("test", "next", [["*e*t", "next"]], [4], 6),
    ("ab", "bc", [["a*", "bd"]], [9], -1),
]
for src, tgt, rules, costs, expected in EXAMPLES:
    print(f"{src!r} -> {tgt!r}, rules={rules}, costs={costs}: kỳ vọng {expected}")

'hello' -> 'world', rules=[['he', 'wo'], ['llo', 'rld']], costs=[3, 4]: kỳ vọng 7
'cat' -> 'dog', rules=[['c*t', 'dog']], costs=[2]: kỳ vọng 3
'test' -> 'next', rules=[['*e*t', 'next']], costs=[4]: kỳ vọng 6
'ab' -> 'bc', rules=[['a*', 'bd']], costs=[9]: kỳ vọng -1


## 3. Kiến thức nền

### 3.1. Khớp mẫu có ký tự đại diện `'*'`

`'*'` khớp **đúng một** ký tự bất kỳ (không phải "nhiều ký tự" như glob). Kiểm tra mẫu `p` tại vị trí `i` của chuỗi `s`: với mọi `j`, `p[j] == '*'` hoặc `p[j] == s[i + j]`.

In [2]:
def match_at(pattern, s, i):
    if i < 0 or i + len(pattern) > len(s):
        return False
    return all(p == "*" or p == s[i + j] for j, p in enumerate(pattern))

print(match_at("c*t", "cat", 0), match_at("c*t", "cut", 0), match_at("c*t", "caat", 0))   # True True False
print(match_at("a*", "xab", 1), match_at("a*", "xab", 2))                                    # True False (vượt biên)
print("c*t".count("*"))                                                                       # 1 dấu *

True True False
True False
1


### 3.2. Biểu thức chính quy: `re.compile`, `.match(s, pos)`

Với mẫu chỉ có chữ thường và `*`, ta đổi `*` thành `.` (khớp một ký tự bất kỳ) và dùng `re.compile(...)`. `compiled.match(s, pos)` chỉ thử khớp **bắt đầu đúng tại `pos`** và trả `None` nếu không khớp. Vì mọi ký tự là chữ thường nên không có chuyện `.` bỏ sót.

In [3]:
import re
rx = re.compile("c.t")                 # từ "c*t"
print(bool(rx.match("cat", 0)), bool(rx.match("xcut", 1)), bool(rx.match("xcut", 0)))   # True True False
print(bool(re.compile("a.").match("xab", 2)))                                              # False: không đủ ký tự

True True False
False


### 3.3. `str.startswith(prefix, start)` và bitset bằng số nguyên Python

- `target.startswith(rep, i)` kiểm tra nhanh (viết bằng C) xem `target[i:i+len(rep)] == rep`.
- **Bitset**: một số nguyên Python lớn dùng như mảng bit: bit `i` bằng 1 nếu điều kiện đúng tại `i`. Phép `&` (và), `>>` (dịch) làm việc trên **cả mảng cùng lúc**: `(mask >> j)` có bit `i` bật ⇔ bit `i + j` của `mask` bật. Lặp qua các bit bật: `low = x & -x` (bit thấp nhất), `i = low.bit_length() - 1`, rồi `x ^= low`.

In [4]:
s = "abcab"
mask_a = sum(1 << i for i, ch in enumerate(s) if ch == "a")     # bit 0 và 3
print(bin(mask_a))
print(bin(mask_a >> 3))                                          # bit 0: s[3] == 'a'

x = 0b10110
out = []
while x:
    low = x & -x
    out.append(low.bit_length() - 1)
    x ^= low
print(out)                                                       # [1, 2, 4]
print("hello".startswith("lo", 3), "hello".startswith("lo", 2))   # True False

0b1001
0b1
[1, 2, 4]
True False


## 4. Ý tưởng nền tảng / trực giác

Tính tay ví dụ 1: `hello → world`. Rule 0 `he → wo` khớp ở `l = 0` (nguồn `"he"`, thay thành `"wo"` đúng bằng `target[0:2]`). Rule 1 `llo → rld` khớp ở `l = 2` (nguồn `"llo"`, thay thành `"rld"` = `target[2:5]`). Hai đoạn `[0,1]` và `[2,4]` **rời nhau** và phủ hết chuỗi: tổng `3 + 4 = 7`.

Ví dụ 4: `source = "ab"`, `target = "bc"`. Luật `a* → bd` khớp ở `l = 0`, nhưng kết quả `"bd" ≠ target[0:2] = "bc"`. Ô đã dùng không thể sửa nữa, nên không có cách nào → `-1`.

Hai điều rút ra:

1. Một lần áp dụng tại `l` **hữu ích** khi: (a) `pattern` khớp `source[l : l+len]`, và (b) `replacement == target[l : l+len]`. Nếu (b) sai thì đoạn đó cuối cùng sẽ sai (ô đã dùng, không sửa được).
2. Các ô **không** được dùng phải thoả `source[i] == target[i]` (miễn phí).

Ta kiểm tra bằng một mô phỏng đúng chữ đề ở mục 5.

## 5. Cách 1 — Brute Force: mô phỏng quá trình (Dijkstra trên chuỗi + mặt nạ ô đã dùng)

**Ý tưởng.** Trạng thái = (chuỗi hiện tại, mặt nạ các ô đã dùng). Từ một trạng thái, thử mọi luật tại mọi `l` thoả điều kiện: đoạn `[l, l+len-1]` chưa dùng, và `pattern` khớp **chuỗi hiện tại**. Chuyển sang trạng thái mới (thay đoạn bằng `replacement`, đánh dấu ô đã dùng) với chi phí `cost + số *`. Dijkstra (chi phí dương) tìm chi phí nhỏ nhất tới một trạng thái có chuỗi bằng `target` (mặt nạ tuỳ ý).

Mô phỏng này **không** dùng nhận xét "ô chưa dùng vẫn bằng source" nên đủ trung thực để kiểm chứng nhận xét đó.

In [5]:
import heapq

def brute_force(source, target, rules, costs):
    n = len(source)
    start = (source, 0)
    best = {start: 0}
    pq = [(0, source, 0)]
    while pq:
        d, cur, used = heapq.heappop(pq)
        if d > best[(cur, used)]:
            continue
        if cur == target:
            return d
        for (pat, rep), c in zip(rules, costs):
            L = len(pat)
            w = c + pat.count("*")
            for l in range(n - L + 1):
                seg_mask = ((1 << L) - 1) << l
                if used & seg_mask:
                    continue                                   # có ô đã dùng
                if all(p == "*" or p == cur[l + j] for j, p in enumerate(pat)):
                    nxt = cur[:l] + rep + cur[l + L:]
                    key = (nxt, used | seg_mask)
                    if d + w < best.get(key, float("inf")):
                        best[key] = d + w
                        heapq.heappush(pq, (d + w, nxt, used | seg_mask))
    return -1

for src, tgt, rules, costs, expected in EXAMPLES:
    print(src, tgt, "->", brute_force(src, tgt, rules, costs), "(kỳ vọng", expected, ")")

hello world -> 7 (kỳ vọng 7 )
cat dog -> 3 (kỳ vọng 3 )
test next -> 6 (kỳ vọng 6 )
ab bc -> -1 (kỳ vọng -1 )


**Độ phức tạp.** Số trạng thái `O(2^n · |Σ|^n)` trong trường hợp xấu — chỉ dùng cho `n ≤ 6` và ít luật. **Nút thắt:** trạng thái chứa cả chuỗi hiện tại, trong khi các ô chưa dùng luôn bằng `source` và ô đã dùng luôn đã đúng `target` (nếu không thì thất bại). Ta không cần nhớ chuỗi, chỉ cần biết "đã xử lý tới đâu".

## 6. Bước đột phá — DP trên chỉ số

Gọi `dp[i]` = chi phí nhỏ nhất để **chuẩn bị xong `i` ký tự đầu**: mỗi ô `< i` hoặc chưa dùng và `source = target`, hoặc nằm trọn trong một đoạn đã áp dụng luật thành công. `dp[0] = 0`.

**Chuyển trạng thái** từ `dp[i]` (nếu hữu hạn):

- **Giữ nguyên ô `i`**: nếu `source[i] == target[i]` thì `dp[i+1] = min(dp[i+1], dp[i])`.
- **Áp dụng luật `r` tại `i`** (độ dài `L`): nếu `i + L ≤ n`, `pattern_r` khớp `source[i:i+L]` và `replacement_r == target[i:i+L]` thì `dp[i+L] = min(dp[i+L], dp[i] + cost_r + stars_r)`.

Đáp án `dp[n]` (hoặc `-1` nếu vô cực).

**Vì sao cạnh chỉ cần "khớp source"?** Ở thời điểm một luật được áp dụng, các ô của nó **chưa dùng**, mà ô chưa dùng thì vẫn là `source` (chưa bị ghi). Vì vậy khớp với chuỗi hiện tại ⇔ khớp `source`.

Đây còn là **đường đi ngắn nhất trên DAG**: đỉnh `0..n`, cạnh `i → i+1` (chi phí 0, nếu `source[i] == target[i]`) và `i → i+L` (chi phí luật). Chỉ số chỉ tăng nên duyệt `i` tăng dần là một thứ tự tô-pô.

## 7. Các cách tiếp theo

- **Cách 2:** DP top-down (đệ quy nhớ) với khớp mẫu bằng vòng lặp.
- **Cách 3:** DP bottom-up với `startswith` + `re` — đáp án nộp.
- **Cách 4:** Dựng danh sách cạnh bằng **bitset**, rồi DP trên cạnh.

### 7.1. Cách 2 — DP top-down

**Giải thích từng đoạn.** `go(i)` = chi phí nhỏ nhất để xử lý **các ô từ `i` tới hết**; `go(n) = 0`. Ta thử giữ nguyên ô `i` (nếu `source[i] == target[i]`) và thử mọi luật tại `i`. Ngược chiều với `dp[i]` ở trên nhưng cùng bản chất.

In [6]:
from functools import lru_cache
import sys
sys.setrecursionlimit(20000)

def min_cost_topdown(source, target, rules, costs):
    n = len(source)
    INF = float("inf")

    @lru_cache(None)
    def go(i):
        if i == n:
            return 0
        best = INF
        if source[i] == target[i]:
            best = go(i + 1)
        for (pat, rep), c in zip(rules, costs):
            L = len(pat)
            if i + L <= n and target[i:i + L] == rep and match_at(pat, source, i):
                best = min(best, c + pat.count("*") + go(i + L))
        return best

    r = go(0)
    return -1 if r == INF else r

for src, tgt, rules, costs, expected in EXAMPLES:
    assert min_cost_topdown(src, tgt, rules, costs) == expected
print("Cách 2 khớp 4 ví dụ")

Cách 2 khớp 4 ví dụ


**Độ phức tạp.** `n` trạng thái × `R` luật × `O(L)` khớp = `O(n · R · L)` = `5000 · 200 · 20 = 2·10^7` phép so ký tự trong trường hợp xấu — đủ chậm trong Python; hơn nữa đệ quy sâu `n = 5000` cần nâng giới hạn đệ quy.

### 7.2. Cách 3 — DP bottom-up với `startswith` và `re` (đáp án nộp)

**Ý tưởng tăng tốc.**

1. Kiểm tra `target.startswith(rep, i)` trước: viết bằng C, loại hầu hết cặp `(i, luật)`.
2. Khớp mẫu bằng `re`: biên dịch một lần mỗi luật (`'*' → '.'`), rồi `rx.match(source, i)` — vòng lặp so ký tự chạy trong C.
3. Duyệt `i` tăng dần, **đẩy** (relax) về phía trước: không đệ quy.
4. Bỏ qua `i` có `dp[i] = INF` ngay.

In [7]:
def min_cost_bottomup(source, target, rules, costs):
    n = len(source)
    INF = float("inf")
    compiled = []
    for (pat, rep), c in zip(rules, costs):
        compiled.append((re.compile(pat.replace("*", ".")), rep, len(pat), c + pat.count("*")))
    dp = [INF] * (n + 1)
    dp[0] = 0
    for i in range(n):
        d = dp[i]
        if d == INF:
            continue
        if source[i] == target[i] and d < dp[i + 1]:
            dp[i + 1] = d
        for rx, rep, L, w in compiled:
            if i + L <= n and target.startswith(rep, i) and rx.match(source, i):
                if d + w < dp[i + L]:
                    dp[i + L] = d + w
    return -1 if dp[n] == INF else dp[n]

for src, tgt, rules, costs, expected in EXAMPLES:
    assert min_cost_bottomup(src, tgt, rules, costs) == expected
print("Cách 3 khớp 4 ví dụ")

Cách 3 khớp 4 ví dụ


**Độ phức tạp.** Về lý thuyết vẫn `O(n · R · L)` nhưng phần nặng chạy trong C (`startswith`, `re`), và `startswith` loại đa số cặp trước khi gọi `re`. Đo thực tế ở mục 11.

### 7.3. Cách 4 — Dựng cạnh bằng bitset, rồi DP trên cạnh

**Ý tưởng.** Với mỗi luật, tính **tập vị trí bắt đầu hợp lệ** chỉ bằng vài phép bit trên các số `n` bit:

- `smask[ch]` có bit `i` ⇔ `source[i] == ch`; `tmask[ch]` tương tự với `target`.
- `valid` ban đầu = các vị trí `0 .. n−L` (đủ chỗ).
- Với mỗi ký tự không phải `'*'` của `pattern` ở vị trí `j`: `valid &= smask[ch] >> j`.
- Với mỗi ký tự của `replacement` ở vị trí `j`: `valid &= tmask[ch] >> j`.

Số phép bit trên mỗi luật chỉ `O(L)` thao tác trên số `n/64` từ máy: tổng chi phí dựng `O(R · L · n/64)`. Sau đó mỗi bit bật là một cạnh `i → i + L` với trọng số `cost + stars`.

In [8]:
def build_edges_bitset(source, target, rules, costs):
    n = len(source)
    smask, tmask = {}, {}
    for i, ch in enumerate(source):
        smask[ch] = smask.get(ch, 0) | (1 << i)
    for i, ch in enumerate(target):
        tmask[ch] = tmask.get(ch, 0) | (1 << i)
    edges = [[] for _ in range(n)]
    for (pat, rep), c in zip(rules, costs):
        L = len(pat)
        if L > n:
            continue
        valid = (1 << (n - L + 1)) - 1                   # vị trí bắt đầu 0..n-L
        for j, ch in enumerate(pat):
            if ch != "*":
                valid &= smask.get(ch, 0) >> j
        for j, ch in enumerate(rep):
            valid &= tmask.get(ch, 0) >> j
        w = c + pat.count("*")
        while valid:
            low = valid & -valid
            i = low.bit_length() - 1
            edges[i].append((i + L, w))
            valid ^= low
    return edges

def min_cost_bitset(source, target, rules, costs):
    n = len(source)
    INF = float("inf")
    edges = build_edges_bitset(source, target, rules, costs)
    dp = [INF] * (n + 1)
    dp[0] = 0
    for i in range(n):
        d = dp[i]
        if d == INF:
            continue
        if source[i] == target[i] and d < dp[i + 1]:
            dp[i + 1] = d
        for j, w in edges[i]:
            if d + w < dp[j]:
                dp[j] = d + w
    return -1 if dp[n] == INF else dp[n]

for src, tgt, rules, costs, expected in EXAMPLES:
    assert min_cost_bitset(src, tgt, rules, costs) == expected
print("Cách 4 khớp 4 ví dụ")
print(build_edges_bitset("hello", "world", EXAMPLES[0][2], EXAMPLES[0][3]))

Cách 4 khớp 4 ví dụ
[[(2, 3)], [], [(5, 4)], [], []]


**Độ phức tạp.** Dựng cạnh `O(R·L·n/w)` (`w = 64`), số cạnh `E ≤ n·R`, DP `O(n + E)`. Không còn so sánh ký tự từng cặp `(i, luật)`.

## 8. TRACE từng bước

Ta in từng bước DP bottom-up cho một ví dụ có cả ô **giữ nguyên** lẫn các lựa chọn luật cạnh tranh: `source = "abcab"`, `target = "abxyb"`, luật `c→x` (3), `ca→xy` (4), `a→y` (1), `c*→xy` (2, có 1 dấu `*`).

| cột | ý nghĩa |
|---|---|
| `i` | số ký tự đã chuẩn bị xong |
| `dp[i]` | chi phí nhỏ nhất đạt được |
| cạnh | ô giữ nguyên (chi phí 0) hoặc luật áp dụng được |

In [9]:
def trace_dp(source, target, rules, costs):
    n = len(source)
    INF = float("inf")
    dp = [INF] * (n + 1)
    dp[0] = 0
    print("source =", source, " target =", target)
    for i in range(n):
        d = dp[i]
        if d == INF:
            print(f"i={i}: dp=INF -> bỏ qua")
            continue
        print(f"i={i}: dp[{i}]={d}")
        if source[i] == target[i]:
            print(f"   giữ nguyên ô {i} ({source[i]}=={target[i]}) -> dp[{i + 1}] ứng viên {d}")
            dp[i + 1] = min(dp[i + 1], d)
        else:
            print(f"   ô {i}: {source[i]}!={target[i]} nên không thể giữ nguyên")
        for k, ((pat, rep), c) in enumerate(zip(rules, costs)):
            L = len(pat)
            if i + L <= n and target[i:i + L] == rep and match_at(pat, source, i):
                w = c + pat.count("*")
                print(f"   luật {k} ({pat}->{rep}) dùng được: +{w} -> dp[{i + L}] ứng viên {d + w}")
                dp[i + L] = min(dp[i + L], d + w)
    print("dp =", dp)
    return -1 if dp[n] == INF else dp[n]

rules_t = [["c", "x"], ["ca", "xy"], ["a", "y"], ["c*", "xy"]]
costs_t = [3, 4, 1, 2]
print("=> đáp án", trace_dp("abcab", "abxyb", rules_t, costs_t))

source = abcab  target = abxyb
i=0: dp[0]=0
   giữ nguyên ô 0 (a==a) -> dp[1] ứng viên 0
i=1: dp[1]=0
   giữ nguyên ô 1 (b==b) -> dp[2] ứng viên 0
i=2: dp[2]=0
   ô 2: c!=x nên không thể giữ nguyên
   luật 0 (c->x) dùng được: +3 -> dp[3] ứng viên 3
   luật 1 (ca->xy) dùng được: +4 -> dp[4] ứng viên 4
   luật 3 (c*->xy) dùng được: +3 -> dp[4] ứng viên 3
i=3: dp[3]=3
   ô 3: a!=y nên không thể giữ nguyên
   luật 2 (a->y) dùng được: +1 -> dp[4] ứng viên 4
i=4: dp[4]=3
   giữ nguyên ô 4 (b==b) -> dp[5] ứng viên 3
dp = [0, 0, 0, 3, 3, 3]
=> đáp án 3


Quan sát: `dp[0..2] = 0` (hai ký tự `ab` đã bằng nhau). Tại `i = 2` có ba cách: `c→x` (3) rồi `a→y` (1) tổng 4; `ca→xy` (4) tổng 4; `c*→xy` (2 + 1 dấu `*` = 3) tổng 3. Chọn 3. Một trace nữa cho ví dụ 4 (`-1`):

In [10]:
print("=> đáp án", trace_dp("ab", "bc", [["a*", "bd"]], [9]))

source = ab  target = bc
i=0: dp[0]=0
   ô 0: a!=b nên không thể giữ nguyên
i=1: dp=INF -> bỏ qua
dp = [0, inf, inf]
=> đáp án -1


## 9. Code nộp LeetCode

In [11]:
from typing import List
import re

class Solution:
    def minimumCost(self, source: str, target: str, rules: List[List[str]], costs: List[int]) -> int:
        n = len(source)
        INF = float("inf")
        compiled = []
        for (pat, rep), c in zip(rules, costs):
            compiled.append((re.compile(pat.replace("*", ".")), rep, len(pat), c + pat.count("*")))

        dp = [INF] * (n + 1)          # dp[i] = chi phí nhỏ nhất để chuẩn bị xong i ký tự đầu
        dp[0] = 0
        for i in range(n):
            d = dp[i]
            if d == INF:
                continue
            if source[i] == target[i] and d < dp[i + 1]:       # giữ nguyên ô i
                dp[i + 1] = d
            for rx, rep, L, w in compiled:                     # áp dụng một luật bắt đầu tại i
                if i + L <= n and target.startswith(rep, i) and rx.match(source, i):
                    if d + w < dp[i + L]:
                        dp[i + L] = d + w
        return -1 if dp[n] == INF else dp[n]

for src, tgt, rules, costs, expected in EXAMPLES:
    assert Solution().minimumCost(src, tgt, rules, costs) == expected
print("Code nộp khớp 4 ví dụ")

Code nộp khớp 4 ví dụ


## 10. Vì sao thuật toán đúng

**(a) Các lần áp dụng rời nhau và độc lập thứ tự.** Mỗi lần áp dụng chỉ ghi lên các ô chưa dùng và sau đó không bao giờ bị đụng lại. Vậy tập các đoạn đã dùng là một họ **đoạn rời nhau**; thứ tự áp dụng không ảnh hưởng.

**(b) Khớp mẫu chỉ phụ thuộc `source`.** Khi một luật được áp dụng ở đoạn `[l, l+L)`, mọi ô của đoạn chưa dùng, nên vẫn là ký tự `source`. Do đó điều kiện "mẫu khớp chuỗi hiện tại" tương đương "mẫu khớp `source[l:l+L]`".

**(c) Điều kiện cho một lần áp dụng "không lãng phí".** Sau khi áp dụng, đoạn này đã là `replacement` và không đổi nữa; để chuỗi cuối bằng `target` cần `replacement == target[l:l+L]`. Ngược lại luật không bao giờ dẫn tới đáp án.

**(d) Ô không dùng.** Giữ nguyên nên cần `source[i] == target[i]`.

**(e) DP đúng.** Mỗi đáp án hợp lệ tương ứng một cách **lát** `0..n−1` bằng các mảnh: mảnh độ dài 1 (ô giữ nguyên, chi phí 0) hoặc mảnh độ dài `L` (một lần áp dụng luật, chi phí `cost + stars`). `dp[i]` là chi phí lát nhỏ nhất của tiền tố `i`. Đây chính xác là đường ngắn nhất trong DAG nên `dp[n]` là đáp án; nếu `dp[n] = INF` thì không có cách lát nào → `-1`.

**(f) Chi phí dương/không âm.** Chi phí mỗi lần áp dụng `≥ 1`; không cần lo chu trình. Dù vậy ta không dùng Dijkstra: DAG đã có thứ tự tô-pô.

## 11. Phân tích Time/Space

Gọi `n = len(source)`, `R = len(rules)`, `L = max độ dài pattern ≤ 20`.

| Cách | Time | Space |
|---|---|---|
| Mô phỏng Dijkstra | cỡ `O(2^n · |Σ|^n · R · n · L)` | cùng bậc |
| DP top-down | `O(n · R · L)` | `O(n)` + đệ quy `O(n)` |
| DP bottom-up (nộp) | `O(n · R · L)` lý thuyết; phần nặng chạy trong C | `O(n + R)` |
| Bitset dựng cạnh + DP | `O(R·L·n/w + n + E)`, `E ≤ n·R` | `O(n·R)` trong trường hợp xấu |

Ta đo trên dữ liệu xấu: `n = 5000`, `R = 200`, mọi luật khớp mọi vị trí (chuỗi toàn `'a'`):

In [12]:
import time, random

def timed(fn, src, tgt, rules, costs):
    t0 = time.time()
    r = fn(src, tgt, rules, costs)
    return r, round(time.time() - t0, 2)

n = 5000
src = tgt = "a" * n
rules = [["a" * 20, "a" * 20] for _ in range(100)] + [["*" * 5 + "a" * 15, "a" * 20] for _ in range(100)]
costs = [random.Random(i).randint(1, 1000) for i in range(200)]
print("toàn 'a', 200 luật khớp mọi nơi:")
print("  nộp (re)   :", timed(Solution().minimumCost, src, tgt, rules, costs))
print("  bitset     :", timed(min_cost_bitset, src, tgt, rules, costs))
try:
    print("  top-down   :", timed(min_cost_topdown, src, tgt, rules, costs))
except RecursionError:
    print("  top-down   : RecursionError (đệ quy sâu n = 5000) -> lý do dùng bottom-up")

rng = random.Random(4)
src2 = "".join(rng.choice("ab") for _ in range(n))
tgt2 = "".join(rng.choice("ab") for _ in range(n))
rules2 = []
for _ in range(200):
    L = rng.randint(1, 20)
    i = rng.randint(0, n - L)
    pat = list(src2[i:i + L])
    for j in rng.sample(range(L), min(L - 1, rng.randint(0, 5))):
        pat[j] = "*"
    rules2.append(["".join(pat), tgt2[i:i + L]])
costs2 = [rng.randint(1, 1000) for _ in range(200)]
print("ngẫu nhiên (luật lấy từ source/target):")
print("  nộp (re)   :", timed(Solution().minimumCost, src2, tgt2, rules2, costs2))
print("  bitset     :", timed(min_cost_bitset, src2, tgt2, rules2, costs2))

toàn 'a', 200 luật khớp mọi nơi:


  nộp (re)   : (0, 0.24)


  bitset     : (0, 0.56)
  top-down   : RecursionError (đệ quy sâu n = 5000) -> lý do dùng bottom-up
ngẫu nhiên (luật lấy từ source/target):
  nộp (re)   : (339330, 0.07)
  bitset     : (339330, 0.02)


## 12. Bộ test

Ví dụ đề, edge case, phản ví dụ cho các cách làm sai, đối chiếu ngẫu nhiên: mô phỏng ↔ top-down ↔ bottom-up ↔ bitset ↔ code nộp.

In [13]:
S = Solution()
for src, tgt, rules, costs, expected in EXAMPLES:
    for fn in (brute_force, min_cost_topdown, min_cost_bottomup, min_cost_bitset, S.minimumCost):
        assert fn(src, tgt, rules, costs) == expected, (fn.__name__, src, tgt)
print("Ví dụ đề: mọi cách khớp")

Ví dụ đề: mọi cách khớp


In [14]:
# Edge case
assert S.minimumCost("a", "a", [["b", "c"]], [5]) == 0                      # đã bằng nhau, không cần luật
assert S.minimumCost("abc", "abc", [["a", "z"]], [1]) == 0                  # không dùng luật nào
assert S.minimumCost("a", "b", [["a", "b"]], [7]) == 7                      # một luật
assert S.minimumCost("a", "b", [["*", "b"]], [7]) == 8                      # '*' khớp, +1
assert S.minimumCost("a", "b", [["c", "b"]], [7]) == -1                     # mẫu không khớp
assert S.minimumCost("abc", "xyz", [["abc", "xyz"], ["a", "x"], ["b", "y"], ["c", "z"]], [10, 1, 1, 1]) == 3   # ba luật nhỏ rẻ hơn một luật lớn
assert S.minimumCost("aa", "bb", [["a", "b"]], [2]) == 4                    # dùng cùng luật hai lần ở hai ô khác nhau
assert S.minimumCost("ab", "ba", [["a", "b"], ["b", "a"]], [1, 1]) == 2
assert S.minimumCost("a", "c", [["a", "b"], ["b", "c"]], [1, 1]) == -1      # không được chuỗi luật chồng lên cùng ô
print("Edge case: OK")

Edge case: OK


**Phản ví dụ** cho các cách nghĩ sai (xem mục 13): bắt buộc phủ mọi ô, quên `+ số '*'`, chỉ kiểm tra mẫu mà bỏ điều kiện `replacement == target`, và cho phép luật **chồng** lên ô đã dùng (chuỗi hoá `a→b→c`, ở trên cho `-1`).

In [15]:
def random_case(rng, n_max=6, alpha="ab"):
    n = rng.randint(1, n_max)
    src = "".join(rng.choice(alpha) for _ in range(n))
    if rng.random() < 0.5:
        # sinh target bằng cách áp dụng ngẫu nhiên vài luật để có nhiều ca có đáp án
        tgt = "".join(rng.choice(alpha) for _ in range(n))
    else:
        tgt = "".join(rng.choice(alpha) for _ in range(n))
    r = rng.randint(1, 4)
    rules, costs = [], []
    for _ in range(r):
        L = rng.randint(1, min(3, n))
        if rng.random() < 0.7:
            i = rng.randint(0, n - L)
            pat = list(src[i:i + L])
            rep = tgt[i:i + L]
        else:
            pat = [rng.choice(alpha) for _ in range(L)]
            rep = "".join(rng.choice(alpha) for _ in range(L))
        stars = rng.sample(range(L), rng.randint(0, L - 1)) if L > 1 else []
        for j in stars:
            pat[j] = "*"
        rules.append(["".join(pat), rep])
        costs.append(rng.randint(1, 6))
    return src, tgt, rules, costs

rng = random.Random(2026)
solved = 0
for it in range(500):
    src, tgt, rules, costs = random_case(rng)
    ref = brute_force(src, tgt, rules, costs)
    assert min_cost_topdown(src, tgt, rules, costs) == ref, (src, tgt, rules, costs)
    assert min_cost_bottomup(src, tgt, rules, costs) == ref, (src, tgt, rules, costs)
    assert min_cost_bitset(src, tgt, rules, costs) == ref, (src, tgt, rules, costs)
    assert S.minimumCost(src, tgt, rules, costs) == ref, (src, tgt, rules, costs)
    solved += ref != -1
print("500 input ngẫu nhiên nhỏ: khớp mô phỏng đề; trong đó có đáp án ≠ -1:", solved)

500 input ngẫu nhiên nhỏ: khớp mô phỏng đề; trong đó có đáp án ≠ -1: 337


In [16]:
# Input vừa: so sánh bốn cách DP với nhau (mô phỏng không với tới)
rng = random.Random(77)
nonneg = 0
for it in range(60):
    n = rng.randint(20, 120)
    src = "".join(rng.choice("abc") for _ in range(n))
    # target = src với một số đoạn bị thay, để thường có lời giải
    tgt = list(src)
    rules, costs = [], []
    for _ in range(rng.randint(3, 15)):
        L = rng.randint(1, 6)
        i = rng.randint(0, n - L)
        rep = "".join(rng.choice("abc") for _ in range(L))
        pat = list(src[i:i + L])
        for j in rng.sample(range(L), rng.randint(0, min(L - 1, 5))):
            pat[j] = "*"
        rules.append(["".join(pat), rep])
        costs.append(rng.randint(1, 20))
        if rng.random() < 0.6:
            tgt[i:i + L] = list(rep)
    tgt = "".join(tgt)
    a = S.minimumCost(src, tgt, rules, costs)
    assert a == min_cost_bitset(src, tgt, rules, costs) == min_cost_topdown(src, tgt, rules, costs) == min_cost_bottomup(src, tgt, rules, costs)
    nonneg += a != -1
print("60 input vừa: các cách khớp nhau; có lời giải:", nonneg)

60 input vừa: các cách khớp nhau; có lời giải: 32


## 13. Lỗi người mới hay gặp

### Lỗi 1 — Bắt buộc mỗi ô phải được một luật phủ

Đề **không** yêu cầu dùng luật trên mọi ô. Ô đã bằng `target` thì cứ để nguyên, miễn phí.

In [17]:
def wrong_must_cover(source, target, rules, costs):
    n = len(source)
    INF = float("inf")
    dp = [INF] * (n + 1); dp[0] = 0
    for i in range(n):
        if dp[i] == INF: continue
        # SAI: thiếu bước "giữ nguyên ô i nếu source[i] == target[i]"
        for (pat, rep), c in zip(rules, costs):
            L = len(pat)
            if i + L <= n and target[i:i + L] == rep and match_at(pat, source, i):
                dp[i + L] = min(dp[i + L], dp[i] + c + pat.count("*"))
    return -1 if dp[n] == INF else dp[n]

args = ("abcab", "abxyb", [["c*", "xy"]], [2])
print("sai :", wrong_must_cover(*args))             # -1
print("đúng:", Solution().minimumCost(*args))        # 3

sai : -1
đúng: 3


**Cách đúng:** thêm cạnh miễn phí `i → i+1` khi `source[i] == target[i]`.

### Lỗi 2 — Quên cộng số dấu `'*'` vào chi phí

In [18]:
def _dp_custom(source, target, rules, costs, star_cost=True, check_rep=True):
    n = len(source)
    INF = float("inf")
    dp = [INF] * (n + 1); dp[0] = 0
    for i in range(n):
        if dp[i] == INF: continue
        if source[i] == target[i]:
            dp[i + 1] = min(dp[i + 1], dp[i])
        for (pat, rep), c in zip(rules, costs):
            L = len(pat)
            if i + L <= n and (not check_rep or target[i:i + L] == rep) and match_at(pat, source, i):
                w = c + (pat.count("*") if star_cost else 0)
                dp[i + L] = min(dp[i + L], dp[i] + w)
    return -1 if dp[n] == INF else dp[n]

src, tgt, rules, costs, _ = EXAMPLES[2]
print("sai :", _dp_custom(src, tgt, rules, costs, star_cost=False))     # 4 thay vì 6
print("đúng:", Solution().minimumCost(src, tgt, rules, costs))           # 6

sai : 4
đúng: 6


**Cách đúng:** `w = cost + pattern.count('*')`.

### Lỗi 3 — Chỉ kiểm tra mẫu khớp mà bỏ điều kiện `replacement == target`

Áp dụng luật xong, ô đã dùng và **không sửa được nữa**. Nếu `replacement` khác `target` ở đoạn đó thì thất bại — ví dụ 4 của đề.

In [19]:
src, tgt, rules, costs, _ = EXAMPLES[3]
print("sai :", _dp_custom(src, tgt, rules, costs, check_rep=False))     # 10 (= 9 + 1 dấu *): tưởng dùng được luật
print("đúng:", Solution().minimumCost(src, tgt, rules, costs))           # -1

sai : 10
đúng: -1


**Cách đúng:** thêm `target[i:i+L] == replacement` (hoặc `target.startswith(rep, i)`).

### Lỗi 4 — Cho phép luật chồng lên ô đã dùng (chuyển chuỗi a → b → c)

Đề cấm dùng lại ô. Nếu ta mô hình hoá như "đồ thị chuỗi" và cho áp dụng luật nhiều lần trên cùng ô, ta nhận kết quả nhỏ hơn đáp án thật.

In [20]:
def wrong_reuse_allowed(source, target, rules, costs):
    # Dijkstra trên CHUỖI (không có mặt nạ ô đã dùng) -> SAI
    n = len(source)
    best = {source: 0}
    pq = [(0, source)]
    while pq:
        d, cur = heapq.heappop(pq)
        if d > best[cur]: continue
        if cur == target: return d
        for (pat, rep), c in zip(rules, costs):
            L = len(pat)
            for l in range(n - L + 1):
                if all(p == "*" or p == cur[l + j] for j, p in enumerate(pat)):
                    nxt = cur[:l] + rep + cur[l + L:]
                    nd = d + c + pat.count("*")
                    if nd < best.get(nxt, float("inf")):
                        best[nxt] = nd
                        heapq.heappush(pq, (nd, nxt))
    return -1

args = ("a", "c", [["a", "b"], ["b", "c"]], [1, 1])
print("sai :", wrong_reuse_allowed(*args))              # 2 (a->b->c) vi phạm 'mỗi ô dùng một lần'
print("đúng:", Solution().minimumCost(*args))            # -1

sai : 2
đúng: -1


**Cách đúng:** mỗi ô chỉ nằm trong **một** đoạn: DP lát đoạn rời nhau trên chỉ số.

## 14. Mẹo & nhận dạng pattern

- **"Mỗi vị trí dùng tối đa một lần" + "độ dài được bảo toàn"** → bài toán lát đoạn rời nhau → DP trên chỉ số / đường ngắn nhất trên DAG.
- **Khớp mẫu có wildcard một ký tự** → vòng lặp `O(L)`, `re` với `.`, hoặc **bitset**: `AND` các `mask[ch] >> j`.
- **Lọc sớm bằng phép kiểm tra rẻ** (`startswith`) trước phép kiểm tra đắt (`re`) giảm hằng số rất nhiều.
- **Phân biệt "ô chưa dùng" và "ô đã dùng":** ô chưa dùng còn nguyên giá trị gốc → có thể kiểm tra mẫu trên `source` thay vì mô phỏng chuỗi hiện tại.
- Bất cứ khi nào đề cho phép áp dụng liên tiếp nhưng cấm chồng lên nhau, hãy thử **đẩy thứ tự áp dụng ra khỏi bài toán**: chỉ còn một "bố cục" các đoạn.
- Trước khi tối ưu, **tự viết mô phỏng đúng chữ đề** (Dijkstra trên (chuỗi, mặt nạ)) để chặn lỗi hiểu đề.

## 15. Biến thể

1. **Khôi phục phương án**: in ra danh sách `(vị trí, luật)` đã dùng.
2. **Đếm số cách tối ưu** (số tập áp dụng khác nhau đạt chi phí nhỏ nhất).
3. **Cho phép ô đã dùng được dùng lại**: bài chuyển thành đường đi ngắn nhất trên không gian chuỗi (rất khác).
4. **Pattern có `'*'` khớp nhiều ký tự**: bài khớp glob, khó hơn.

Ta cài biến thể 1 (khôi phục phương án) và **kiểm định phương án** bằng cách mô phỏng thật sự theo đề.

In [21]:
def min_cost_with_plan(source, target, rules, costs):
    n = len(source)
    INF = float("inf")
    dp = [INF] * (n + 1); dp[0] = 0
    par = [None] * (n + 1)                     # par[j] = (i, chỉ số luật hoặc None nếu giữ nguyên)
    for i in range(n):
        if dp[i] == INF: continue
        if source[i] == target[i] and dp[i] < dp[i + 1]:
            dp[i + 1] = dp[i]; par[i + 1] = (i, None)
        for k, ((pat, rep), c) in enumerate(zip(rules, costs)):
            L = len(pat)
            if i + L <= n and target[i:i + L] == rep and match_at(pat, source, i):
                w = c + pat.count("*")
                if dp[i] + w < dp[i + L]:
                    dp[i + L] = dp[i] + w; par[i + L] = (i, k)
    if dp[n] == INF:
        return -1, []
    plan, j = [], n
    while j > 0:
        i, k = par[j]
        if k is not None:
            plan.append((i, k))
        j = i
    return dp[n], plan[::-1]

def apply_plan(source, target, rules, costs, plan):
    """Mô phỏng đúng luật đề: kiểm tra từng lần áp dụng hợp lệ, trả về (chuỗi cuối, tổng chi phí)."""
    cur, used, total = list(source), [False] * len(source), 0
    for l, k in plan:
        pat, rep = rules[k]
        L = len(pat)
        assert not any(used[l:l + L]), "ô đã dùng"
        assert all(p == "*" or p == cur[l + j] for j, p in enumerate(pat)), "mẫu không khớp"
        cur[l:l + L] = list(rep)
        for j in range(L): used[l + j] = True
        total += costs[k] + pat.count("*")
    return "".join(cur), total

src, tgt, rules, costs, _ = EXAMPLES[0]
print(min_cost_with_plan(src, tgt, rules, costs))
rng = random.Random(15)
for _ in range(300):
    src, tgt, rules, costs = random_case(rng)
    cost, plan = min_cost_with_plan(src, tgt, rules, costs)
    assert cost == S.minimumCost(src, tgt, rules, costs)
    if cost != -1:
        final, total = apply_plan(src, tgt, rules, costs, plan)
        assert final == tgt and total == cost
print("Biến thể khôi phục phương án: phương án mô phỏng đúng và đạt chi phí tối ưu")

(7, [(0, 0), (2, 1)])
Biến thể khôi phục phương án: phương án mô phỏng đúng và đạt chi phí tối ưu


## 16. So sánh các cách

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Mô phỏng Dijkstra | (chuỗi, mặt nạ ô đã dùng) | mũ | mũ | trọng tài, `n ≤ 6` |
| 2. DP top-down | `go(i)` đệ quy nhớ | `O(n·R·L)` | `O(n)` | dễ viết, `n` vừa |
| 3. DP bottom-up (nộp) | đẩy `dp[i]`, `startswith` + `re` | `O(n·R·L)` (phần nặng trong C) | `O(n+R)` | thi đấu |
| 4. Bitset + DP | `AND` các `mask >> j` dựng cạnh | `O(R·L·n/w + n + E)` | `O(E)` | `R·L` lớn, muốn tránh so sánh từng cặp |

## 17. Bài tập (5 bài, từ dễ đến khó)

**Bài 1 (đoán output).** `match_at("a*c", "abc", 0)`, `match_at("a*c", "abd", 0)`, `match_at("ab", "xab", 1)` bằng gì?

**Bài 2 (tính tay).** `source = "ab"`, `target = "cd"`, `rules = [["a*","cd"], ["a","c"], ["b","d"]]`, `costs = [5, 1, 1]`. Đáp án là bao nhiêu và dùng luật nào?

**Bài 3 (điền TODO).** Hoàn thiện `usable(pattern, replacement, source, target, i)` trả `True` nếu luật dùng được tại `i` (đủ chỗ, mẫu khớp `source`, `replacement` bằng đoạn `target`).

**Bài 4 (tự viết lại).** Viết `my_solve(source, target, rules, costs)` bằng DP bottom-up dùng `usable`.

**Bài 5 (biến thể / mini challenge).** Viết `count_optimal(source, target, rules, costs)` trả về `(chi_phí_nhỏ_nhất, số_cách_tối_ưu)` — số tập `{(vị trí, chỉ số luật)}` khác nhau đạt chi phí nhỏ nhất. Kiểm chứng bằng đệ quy liệt kê mọi cách lát.

In [22]:
# Bài 3 — TODO (không gọi khi chạy notebook)
def usable(pattern, replacement, source, target, i):
    # TODO 1: nếu i + len(pattern) > len(source) -> False
    # TODO 2: target[i:i+L] phải bằng replacement
    # TODO 3: với mỗi j, pattern[j] == '*' hoặc pattern[j] == source[i + j]
    pass

# Bài 4 — TODO
def my_solve(source, target, rules, costs):
    # TODO: dp[0] = 0; duyệt i; giữ nguyên nếu source[i] == target[i]; thử mọi luật với usable
    pass

# Bài 5 — TODO
def count_optimal(source, target, rules, costs):
    # TODO: dp[i] = (chi phí, số cách); cộng số cách khi bằng nhau
    pass

## 18. Gợi ý 3 mức

**Bài 1.**
- Mức 1: `'*'` khớp đúng một ký tự bất kỳ.
- Mức 2: so từng vị trí: `a=a`, `*` bất kỳ, `c` với `c` hoặc `d`.
- Mức 3: `True`, `False`, `True`.

**Bài 2.**
- Mức 1: hai cách: một luật `a*→cd` (5 + 1 sao = 6) hoặc hai luật nhỏ.
- Mức 2: `a→c` (1) và `b→d` (1) rời nhau.
- Mức 3: đáp án `2`.

**Bài 3.**
- Mức 1: kiểm tra biên trước tiên.
- Mức 2: `target[i:i+L] == replacement`.
- Mức 3: `all(p == '*' or p == source[i+j] for j, p in enumerate(pattern))`.

**Bài 4.**
- Mức 1: `INF = float('inf')`, `dp = [INF]*(n+1)`.
- Mức 2: cạnh giữ nguyên + cạnh luật.
- Mức 3: `return -1 if dp[n] == INF else dp[n]`.

**Bài 5.**
- Mức 1: `dp[i]` là cặp `(chi phí, số cách)`.
- Mức 2: khi tìm được chi phí nhỏ hơn thì **thay** số cách; khi bằng thì **cộng** số cách.
- Mức 3: cạnh "giữ nguyên" và cạnh luật đều có thể đồng thời tối ưu; số cách nhân với số cách của `dp[i]` (cạnh chỉ thêm một lựa chọn nên cộng thẳng `ways[i]`).

## 19. Đáp án tham khảo

In [23]:
# Bài 1
assert match_at("a*c", "abc", 0) and not match_at("a*c", "abd", 0) and match_at("ab", "xab", 1)

# Bài 2
assert Solution().minimumCost("ab", "cd", [["a*", "cd"], ["a", "c"], ["b", "d"]], [5, 1, 1]) == 2

# Bài 3
def usable_ans(pattern, replacement, source, target, i):
    L = len(pattern)
    if i + L > len(source):
        return False
    if target[i:i + L] != replacement:
        return False
    return all(p == "*" or p == source[i + j] for j, p in enumerate(pattern))

assert usable_ans("c*", "xy", "abcab", "abxyb", 2)
assert not usable_ans("c*", "xy", "abcab", "abxyb", 3)
assert not usable_ans("c*", "xy", "abcab", "abxzb", 2)

# Bài 4
def my_solve_ans(source, target, rules, costs):
    n = len(source)
    INF = float("inf")
    dp = [INF] * (n + 1)
    dp[0] = 0
    for i in range(n):
        if dp[i] == INF:
            continue
        if source[i] == target[i]:
            dp[i + 1] = min(dp[i + 1], dp[i])
        for (pat, rep), c in zip(rules, costs):
            if usable_ans(pat, rep, source, target, i):
                dp[i + len(pat)] = min(dp[i + len(pat)], dp[i] + c + pat.count("*"))
    return -1 if dp[n] == INF else dp[n]

rng = random.Random(61)
for _ in range(300):
    src, tgt, rules, costs = random_case(rng)
    assert my_solve_ans(src, tgt, rules, costs) == S.minimumCost(src, tgt, rules, costs)
print("Bài 1–4 OK")

Bài 1–4 OK


In [24]:
# Bài 5: đếm số cách tối ưu
def count_optimal_ans(source, target, rules, costs):
    n = len(source)
    INF = float("inf")
    dp = [(INF, 0)] * (n + 1)
    dp[0] = (0, 1)
    def relax(j, cost, ways):
        c0, w0 = dp[j]
        if cost < c0:
            dp[j] = (cost, ways)
        elif cost == c0:
            dp[j] = (c0, w0 + ways)
    for i in range(n):
        c, w = dp[i]
        if c == INF:
            continue
        if source[i] == target[i]:
            relax(i + 1, c, w)
        for (pat, rep), cs in zip(rules, costs):
            if usable_ans(pat, rep, source, target, i):
                relax(i + len(pat), c + cs + pat.count("*"), w)
    return (-1, 0) if dp[n][0] == INF else dp[n]

def count_optimal_brute(source, target, rules, costs):
    n = len(source)
    results = []
    def rec(i, cost):
        if i == n:
            results.append(cost); return
        if source[i] == target[i]:
            rec(i + 1, cost)
        for (pat, rep), c in zip(rules, costs):
            if usable_ans(pat, rep, source, target, i):
                rec(i + len(pat), cost + c + pat.count("*"))
    rec(0, 0)
    if not results:
        return (-1, 0)
    m = min(results)
    return (m, results.count(m))

assert count_optimal_ans("ab", "cd", [["a", "c"], ["b", "d"], ["a", "c"]], [1, 1, 1]) == (2, 2)   # hai luật trùng nhau -> hai cách
rng = random.Random(62)
for _ in range(300):
    src, tgt, rules, costs = random_case(rng)
    assert count_optimal_ans(src, tgt, rules, costs) == count_optimal_brute(src, tgt, rules, costs), (src, tgt, rules, costs)
print("Bài 5 OK")

Bài 5 OK


## 20. Template nhớ nhanh + Checklist trước khi submit

**Template "lát đoạn rời nhau bằng DP chỉ số":**

```python
dp = [INF] * (n + 1); dp[0] = 0
for i in range(n):
    if dp[i] == INF: continue
    if source[i] == target[i]:                         # ô giữ nguyên, miễn phí
        dp[i + 1] = min(dp[i + 1], dp[i])
    for rx, rep, L, w in compiled:                     # w = cost + số '*'
        if i + L <= n and target.startswith(rep, i) and rx.match(source, i):
            dp[i + L] = min(dp[i + L], dp[i] + w)
return -1 if dp[n] == INF else dp[n]
```

**Checklist trước khi submit:**

- [ ] Mẫu `*` khớp **đúng một** ký tự; đổi thành `.` khi dùng `re`.
- [ ] Điều kiện áp dụng luật: đủ chỗ, mẫu khớp `source`, `replacement == target[i:i+L]`.
- [ ] Chi phí luật = `cost + pattern.count('*')`.
- [ ] Có cạnh giữ nguyên ô khi `source[i] == target[i]` (miễn phí).
- [ ] Không cho luật chồng lên ô đã dùng (đoạn rời nhau).
- [ ] `dp[n] == INF` → `-1`; đã đối chiếu mô phỏng đề trên input nhỏ.